# Expert-CFG: VQA-RAD Ablation (4 runs)

| Run | γ | LoRA | Attn |
|-----|---|------|------|
| 1_base | 1.0 | N | N |
| 2_cfg | 1.5 | N | N |
| 3_lora | 1.0 | Y | N |
| 4_full | 1.5 | Y | Y |

**필요한 Kaggle 데이터셋:**
```
/kaggle/input/expert-cfg-data/
├── VQA-RAD/images/
└── test.json

/kaggle/input/datasets/ciderchicken/expert-cfg-weights/
└── Phi-3.5-vision-instruct/   # 모델 가중치 (.safetensors 등)
```

## Step 1. 패키지 설치

In [ ]:
!pip install --prefer-binary transformers==4.43.0 tokenizers==0.19.1 accelerate peft bitsandbytes -q

## Step 2. Expert-CFG 클론

In [ ]:
import os, subprocess, sys, json

if not os.path.exists('/kaggle/working/Expert-CFG'):
    subprocess.run([
        'git', 'clone',
        'https://github.com/minseongkim-dev/expert-cfg',
        '/kaggle/working/Expert-CFG'
    ], check=True)
    print('클론 완료')
else:
    print('이미 존재')

## Step 3. 모델 가중치 복사

레포 안의 `Phi-3.5-vision-instruct/`에는 커스텀 processor(`processing_phi3_v_cfg.py`)가 있음.  
Kaggle 데이터셋의 가중치(.safetensors 등)를 그 폴더로 복사해서 합침.

In [ ]:
import os, shutil, glob

WEIGHTS_SRC = '/kaggle/input/datasets/ciderchicken/expert-cfg-weights/Phi-3.5-vision-instruct'
MODEL_DIR   = '/kaggle/working/Expert-CFG/Phi-3.5-vision-instruct'

# Rebuild MODEL_DIR fresh each time so patches are applied cleanly
if os.path.exists(MODEL_DIR):
    shutil.rmtree(MODEL_DIR)
os.makedirs(MODEL_DIR, exist_ok=True)

# Copy repo's custom processor files first
REPO_DIR = '/kaggle/working/Expert-CFG/Phi-3.5-vision-instruct'
# (MODEL_DIR == REPO_DIR — files already there from clone, already removed above)
# Copy custom files from repo backup if needed (handled by git clone earlier)

# Copy weights from Kaggle dataset
for fname in os.listdir(WEIGHTS_SRC):
    src = os.path.join(WEIGHTS_SRC, fname)
    dst = os.path.join(MODEL_DIR, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        print(f'복사: {fname}')

# Copy custom processor from repo (overwrite if weights had a different version)
REPO_PROC_SRC = '/kaggle/working/Expert-CFG'
for proc_file in ['processing_phi3_v_cfg.py', 'processor_config.json']:
    # These are already in MODEL_DIR from weights copy or will be restored below
    pass

print(f'\n모델 디렉토리 파일 수: {len(os.listdir(MODEL_DIR))}')

# Patch modeling_phi3_v.py for new DynamicCache API (transformers >= 4.44)
target = os.path.join(MODEL_DIR, 'modeling_phi3_v.py')
if os.path.exists(target):
    with open(target) as f:
        content = f.read()
    content = content.replace('past_key_values.seen_tokens',        'past_key_values.get_seq_length()')
    content = content.replace('.get_usable_length(seq_length)',      '.get_seq_length()')
    content = content.replace('.get_usable_length(kv_seq_len, self.layer_idx)', '.get_seq_length(self.layer_idx)')
    with open(target, 'w') as f:
        f.write(content)
    print('modeling_phi3_v.py 패치 완료')
else:
    print('⚠️  modeling_phi3_v.py 없음 — weights 경로 확인 필요')

# Delete HF module cache so trust_remote_code picks up patched file
for d in glob.glob('/root/.cache/huggingface/modules/transformers_modules/*Phi*'):
    shutil.rmtree(d)
    print(f'캐시 삭제: {d}')
print('캐시 정리 완료')

## Step 4. 경로 확인

In [ ]:
import sys, json

DATA_ROOT = '/kaggle/input/expert-cfg-data'
TEST_JSON = f'{DATA_ROOT}/test.json'
IMG_ROOT  = f'{DATA_ROOT}/VQA-RAD/images'
WORK_DIR  = '/kaggle/working'
LORA_DIR  = f'{WORK_DIR}/lora_weights/logs_phi35_pubmed_instruct'

for label, path in [('model', MODEL_DIR), ('test.json', TEST_JSON), ('images/', IMG_ROOT), ('lora_dir', LORA_DIR)]:
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {label}: {path}")

if os.path.exists(IMG_ROOT):
    imgs = [f for f in os.listdir(IMG_ROOT) if f.lower().endswith(('.jpg','.png'))]
    print(f'   이미지 수: {len(imgs)}개')

if os.path.exists(MODEL_DIR):
    print(f'   모델 파일: {sorted(os.listdir(MODEL_DIR))}')

In [ ]:
## Step 4b. LoRA 가중치 다운로드 (Run 3, 4 에서 사용)
import subprocess

os.makedirs(LORA_DIR, exist_ok=True)

# ecoxial2007/Phi-3.5V-Med 에서 adapter_config.json + adapter_model.safetensors 다운로드
lora_files = ['adapter_config.json', 'adapter_model.safetensors']
hf_repo = 'ecoxial2007/Phi-3.5V-Med'

missing = [f for f in lora_files if not os.path.exists(os.path.join(LORA_DIR, f))]
if not missing:
    print('LoRA 가중치 이미 존재, 다운로드 생략')
else:
    print(f'다운로드 필요: {missing}')
    for fname in missing:
        url = f'https://huggingface.co/{hf_repo}/resolve/main/{fname}'
        dst = os.path.join(LORA_DIR, fname)
        result = subprocess.run(['wget', '-q', '--show-progress', url, '-O', dst], capture_output=True, text=True)
        if result.returncode == 0:
            print(f'✅ {fname}')
        else:
            print(f'❌ {fname}: {result.stderr[:200]}')

print(f'\nLoRA 디렉토리 내용: {os.listdir(LORA_DIR)}')

## Step 5. 4가지 실행

> 각 셀을 순서대로 실행. 셀당 약 1-2시간.

In [ ]:
# Run 1: baseline (γ=1.0, no LoRA, no Attn) — 4비트 양자화로 T4 메모리 절약
import subprocess, sys

# 각 실행 전 HF 캐시 삭제 (trust_remote_code가 패치된 파일 사용하도록)
import glob, shutil
for d in glob.glob('/root/.cache/huggingface/modules/transformers_modules/*Phi*'):
    shutil.rmtree(d)

cmd = [
    sys.executable, 'demo.py',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',          # 4비트 양자화 (LoRA adapter 없이)
    '--bf16',
    '--save_path',          f'{WORK_DIR}/result_1_base.json',
]
print('▶ Run 1: base (4bit, no LoRA)')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 2: CFG only (γ=1.5, no LoRA, no Attn)
for d in glob.glob('/root/.cache/huggingface/modules/transformers_modules/*Phi*'):
    shutil.rmtree(d)

cmd = [
    sys.executable, 'demo.py',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',          # 4비트 양자화 (LoRA adapter 없이)
    '--cfg',                '1.5',
    '--bf16',
    '--save_path',          f'{WORK_DIR}/result_2_cfg.json',
]
print('▶ Run 2: CFG only (4bit, γ=1.5)')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 3: LoRA only (γ=1.0, LoRA, no Attn)
for d in glob.glob('/root/.cache/huggingface/modules/transformers_modules/*Phi*'):
    shutil.rmtree(d)

cmd = [
    sys.executable, 'demo.py',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--output_dir',         LORA_DIR,
    '--bf16',
    '--save_path',          f'{WORK_DIR}/result_3_lora.json',
]
print('▶ Run 3: LoRA only')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

In [ ]:
# Run 4: Expert-CFG full (γ=1.5, LoRA, Attn)
for d in glob.glob('/root/.cache/huggingface/modules/transformers_modules/*Phi*'):
    shutil.rmtree(d)

cmd = [
    sys.executable, 'demo.py',
    '--dataset',            'vqa-rad',
    '--input_json',         TEST_JSON,
    '--img_root',           IMG_ROOT,
    '--model_name_or_path', MODEL_DIR,
    '--use_qlora',
    '--output_dir',         LORA_DIR,
    '--cfg',                '1.5',
    '--attn',               '3.0',
    '--use_attn_patch',
    '--bf16',
    '--save_path',          f'{WORK_DIR}/result_4_full.json',
]
print('▶ Run 4: Expert-CFG full')
subprocess.run(cmd, cwd='/kaggle/working/Expert-CFG')

## Step 6. 결과 비교

In [ ]:
runs = [
    ('1_base', f'{WORK_DIR}/result_1_base.json', '1.0', 'N', 'N'),
    ('2_cfg',  f'{WORK_DIR}/result_2_cfg.json',  '1.5', 'N', 'N'),
    ('3_lora', f'{WORK_DIR}/result_3_lora.json', '1.0', 'Y', 'N'),
    ('4_full', f'{WORK_DIR}/result_4_full.json', '1.5', 'Y', 'Y'),
]

print(f"{'Run':<30} {'Closed':>8} {'Open':>8} {'Overall':>9}  γ  LoRA  Attn")
print('-' * 75)

for name, path, cfg, lora, attn in runs:
    if not os.path.exists(path):
        print(f"{'vqa-rad_' + name:<30} {'(미실행)':>8}")
        continue
    with open(path) as f:
        out = json.load(f)
    s = out['summary']
    closed_acc = s.get('closed_accuracy', 0) * 100
    open_acc   = s.get('open_accuracy', 0) * 100
    overall    = s['accuracy'] * 100
    print(f"{'vqa-rad_' + name:<30} {closed_acc:>7.1f}%  {open_acc:>6.1f}%  {overall:>7.1f}%  {cfg}   {lora}     {attn}")